# lobster — walkthrough

Build a limit order book, run a multi-agent simulation, and analyze the
result (spread, mid path, market-maker P&L, and adverse selection).

```bash
pip install -e ".[dev,plot]"
```


## 1. Build a book by hand


In [ ]:
from lobster import Order, OrderBook, Side

book = OrderBook()
book.add(Order(Side.BUY, 100, 99.5))
book.add(Order(Side.SELL, 50, 100.5))
book.snapshot()

`mid`, `spread`, and the size-weighted `microprice` are all live properties.


## 2. Run a multi-agent simulation

Noise traders supply random flow (some marketable), a momentum trader
chases the tape, and a market maker quotes both sides with inventory skew.


In [ ]:
from lobster import Simulation
from lobster.agents import MarketMakerAgent, MomentumAgent, NoiseAgent
from lobster.analytics import Analytics

sim = Simulation(agents=[
    NoiseAgent(agent_id=1, intensity=0.6, market_order_rate=0.25),
    NoiseAgent(agent_id=2, intensity=0.5, market_order_rate=0.25),
    MomentumAgent(agent_id=3, lookback=20, threshold=0.35),
    MarketMakerAgent(agent_id=4, half_spread=0.4, qty=12),
], seed=7)
for _ in sim.run(3000):
    pass

an = Analytics(metrics=sim.metrics, tape=sim.tape, agents=sim.agents)
an.spread_stats()

## 3. Plot the mid-price path


In [ ]:
import matplotlib.pyplot as plt

mids = [m.mid for m in sim.metrics if m.mid is not None]
plt.figure(figsize=(9, 4))
plt.plot(mids, lw=0.8)
plt.title('Mid price over the simulation')
plt.xlabel('step'); plt.ylabel('mid')
plt.tight_layout()
plt.show()

## 4. Market-maker P&L and adverse selection

A market maker earns the spread but is *adversely selected* — the price
tends to move against it right after it fills. `markout` measures exactly
that (negative = adverse).


In [ ]:
print('MM markout @10:', round(an.markout(4, horizon=10), 4))
for aid, pnl in an.agent_pnl().items():
    print(aid, {k: round(v, 2) for k, v in pnl.items()})

## 5. Replaying real LOBSTER data

The same book reconstructs from a NASDAQ-style message stream:


In [ ]:
from lobster import replay_csv

real_book = replay_csv('../data/sample_messages.csv', price_scale=1e-4)
real_book.snapshot()